# 18.10 综合实践二：独立编辑事件与多环境因果验证

《统计遗传与智能育种学》第十八章 · 电子资源 `18b_edit_validation.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：18b_edit_validation.ipynb。实践模拟16个独立事件（8个目标编辑、8个对照）、3个环境以及每个事件×环境6个植株重复。分析先得到事件×环境均值，再按独立事件进行分层的聚类自助法（cluster bootstrap），估计参考环境的编辑效应及编辑×环境互作；避免把同一事件内部重复错误当成新的独立编辑。

运行检查：重启内核并运行全部单元。本书实际运行中，参考环境的编辑效应为0.619，事件级自助法95%区间为0.242～0.984；环境1相对参考环境的互作项为−0.391。若只报告表现最好的编辑事件，表面差异由0.555放大到0.943。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1802；16个独立事件；3个环境；每个事件×环境6个重复；48个事件—环境单元、288个植株观测；3000次事件级自助法；无外部数据文件。

In [1]:
import sys
import numpy as np
import pandas as pd

SEED = 1802
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('numpy', np.__version__, 'pandas', pd.__version__)
print('SEED', SEED)

n_events, n_rep = 16, 6
env_effect = np.array([0.0, 0.55, -0.25])
edit_by_env = np.array([0.0, -0.35, 0.18])
rows = []
for event in range(n_events):
    edit = int(event < n_events // 2)
    event_effect = rng.normal(0, .32)
    molecular_shift = 0.72 * edit + rng.normal(0, .08)
    for env in range(3):
        for rep in range(n_rep):
            trait = (5.0 + 0.68 * edit + env_effect[env]
                     + edit * edit_by_env[env] + event_effect
                     + rng.normal(0, .48))
            rows.append((event, edit, env, rep, molecular_shift, trait))
df = pd.DataFrame(rows, columns=[
    'event', 'edit', 'environment', 'rep', 'molecular_shift', 'trait'])
cell = df.groupby(['event', 'edit', 'environment'], as_index=False).agg(
    trait=('trait', 'mean'), molecular_shift=('molecular_shift', 'mean'))

def design(frame):
    e1 = (frame.environment.to_numpy() == 1).astype(float)
    e2 = (frame.environment.to_numpy() == 2).astype(float)
    ed = frame.edit.to_numpy(dtype=float)
    return np.column_stack([np.ones(len(frame)), ed, e1, e2, ed * e1, ed * e2])

names = ['intercept', 'edit', 'env1', 'env2', 'edit_x_env1', 'edit_x_env2']
beta = np.linalg.lstsq(design(cell), cell.trait.to_numpy(), rcond=None)[0]
boot = []
edit_ids = cell.loc[cell.edit == 1, 'event'].unique()
ctrl_ids = cell.loc[cell.edit == 0, 'event'].unique()
for _ in range(3000):
    pieces = []
    new_id = 0
    for ids in (edit_ids, ctrl_ids):
        for old_id in rng.choice(ids, size=len(ids), replace=True):
            z = cell[cell.event == old_id].copy()
            z['event'] = new_id
            new_id += 1
            pieces.append(z)
    b = pd.concat(pieces, ignore_index=True)
    boot.append(np.linalg.lstsq(design(b), b.trait.to_numpy(), rcond=None)[0])
boot = np.asarray(boot)
ci = np.quantile(boot, [.025, .975], axis=0)
result = pd.DataFrame({'estimate': beta, 'ci_low': ci[0], 'ci_high': ci[1]}, index=names)
print('independent_events', n_events, 'event_environment_cells', len(cell), 'raw_observations', len(df))
print(result.round(3).to_string())

event_mean = df.groupby(['event', 'edit'], as_index=False).trait.mean()
mean_edit = event_mean.loc[event_mean.edit == 1, 'trait'].mean()
mean_ctrl = event_mean.loc[event_mean.edit == 0, 'trait'].mean()
best_edit = event_mean.loc[event_mean.edit == 1, 'trait'].max()
print('all_event_mean_difference', round(mean_edit - mean_ctrl, 3))
print('best_edited_event_minus_control_mean', round(best_edit - mean_ctrl, 3))
print('mean_molecular_shift edited/control',
      np.round(cell.groupby('edit').molecular_shift.mean().to_numpy(), 3))


Python 3.12.13
numpy 2.3.5 pandas 2.2.3
SEED 1802


independent_events 16 event_environment_cells 48 raw_observations 288
             estimate  ci_low  ci_high
intercept       4.991   4.716    5.249
edit            0.619   0.242    0.984
env1            0.588   0.417    0.757
env2           -0.193  -0.463    0.060
edit_x_env1    -0.391  -0.675   -0.125
edit_x_env2     0.201  -0.113    0.531
all_event_mean_difference 0.555
best_edited_event_minus_control_mean 0.943
mean_molecular_shift edited/control [0.043 0.715]


结果解释：增加同一编辑事件内的重复植株主要提高该事件均值的估计精度，并不会增加独立编辑事件数。事件级自助法保留了同一事件跨环境观测的相关结构；若在多个随机波动结果中事后选取效应最大的编辑事件，则效应估计会产生选择偏倚并倾向于被高估。

可修改练习：把每组独立事件从8个减少到3个、但把每个事件重复增加到24个；比较自助法区间。再把edit_by_env全部设为0，观察互作区间。